## Step 1: Install PySpark
This cell installs the Apache Spark library to handle our large-scale data processing directly in the Colab environment.

In [1]:
!pip install pyspark


## Step 2: Download and Unzip the Citi Bike Dataset
Here we download the public Jersey City (JC) January 2020 Citi Bike trip history zip file and extract the raw CSV file to a local directory.

In [2]:
import urllib.request
import zipfile
import os

# Using the Jersey City (JC) January 2020 data which is publicly available
url = 'https://s3.amazonaws.com/tripdata/JC-202001-citibike-tripdata.csv.zip'
zip_path = 'JC-202001-citibike-tripdata.csv.zip'

print("Downloading Citi Bike dataset...")
urllib.request.urlretrieve(url, zip_path)
print("Download complete.")

print("Unzipping dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('citibike_data')
print("Extraction complete. Files extracted to 'citibike_data/' directory:")
print(os.listdir('citibike_data'))

Download complete.
Unzipping dataset...
Extraction complete. Files extracted to 'citibike_data/' directory:
['JC-202001-citibike-tripdata.csv']


## Step 3: Initialize Spark and Verify Raw Data Loading
We establish a Spark Session to load and verify the unzipped Citi Bike CSV file to confirm our environment can read the schema.

In [3]:
from pyspark.sql import SparkSession

# Start a simple Spark session to verify we can load the extracted CSV
spark = SparkSession.builder.appName("CitiBikeAnalysis").getOrCreate()

# Updated path matching the extracted Jersey City file name
csv_path = 'citibike_data/JC-202001-citibike-tripdata.csv'
df = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"Loaded {df.count()} rows. Here is a sample:")
df.show(5)

Loaded 26020 rows. Here is a sample:
+------------+--------------------+--------------------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+----------+----------+------+
|tripduration|           starttime|            stoptime|start station id|start station name|start station latitude|start station longitude|end station id|end station name|end station latitude|end station longitude|bikeid|  usertype|birth year|gender|
+------------+--------------------+--------------------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+----------+----------+------+
|         226|2020-01-01 00:04:...|2020-01-01 00:08:...|            3186|     Grove St PATH|     40.71958611647166|     -74.04311746358871|          3211|      Newark Ave|         40.72152515|        -74.046304

## Step 4: Data Quality Cleaning & Feature Engineering
In this block, we parse start and stop timestamps, calculate trip duration in minutes, filter out statistical outliers (trips under 1 minute or over 4 hours), convert numerical gender labels to readable categories, and drop null key values.

In [4]:
from pyspark.sql.functions import col, when, to_timestamp, round
from pyspark.sql import SparkSession
import urllib.request
import zipfile
import os

# 1. Automatically re-download if the file is missing
csv_dir = 'citibike_data'
csv_filename = 'JC-202001-citibike-tripdata.csv'
local_csv_path = os.path.abspath(os.path.join(csv_dir, csv_filename))

if not os.path.exists(local_csv_path):
    print("Dataset not found. Re-downloading...")
    url = 'https://s3.amazonaws.com/tripdata/JC-202001-citibike-tripdata.csv.zip'
    zip_path = 'JC-202001-citibike-tripdata.csv.zip'
    urllib.request.urlretrieve(url, zip_path)

    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(csv_dir)
    print("Download and extraction completed.")

# 2. Ensure SparkSession is initialized and read the CSV
spark = SparkSession.builder.appName("CitiBikeAnalysis").getOrCreate()
df = spark.read.csv(f"file://{local_csv_path}", header=True, inferSchema=True)

# 3. Convert starttime and stoptime to proper timestamp types
cleaned_df = df \
    .withColumn("starttime", to_timestamp(col("starttime"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("stoptime", to_timestamp(col("stoptime"), "yyyy-MM-dd HH:mm:ss"))

# 4. Calculate trip duration in minutes for easier analysis and filter outliers
cleaned_df = cleaned_df \
    .withColumn("duration_minutes", round(col("tripduration") / 60, 2)) \
    .filter((col("duration_minutes") >= 1.0) & (col("duration_minutes") <= 240.0))

# 5. Map numerical gender representations to human-readable strings
cleaned_df = cleaned_df.withColumn(
    "gender_label",
    when(col("gender") == 1, "Male")
    .when(col("gender") == 2, "Female")
    .otherwise("Unknown")
)

# 6. Remove records with missing essential values
cleaned_df = cleaned_df.dropna(subset=["start station name", "end station name"])

# 7. Display the cleaned dataset schema and sample rows
cleaned_df.printSchema()

print(f"Original rows: {df.count()}")
print(f"Cleaned rows: {cleaned_df.count()}")

display(cleaned_df.select("starttime", "stoptime", "duration_minutes", "start station name", "end station name", "usertype", "gender_label").limit(5).toPandas())

root
 |-- tripduration: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: integer (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: integer (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- gender_label: string (nullable = false)

Original rows: 26020
Cleaned rows: 25997


,starttime,stoptime,duration_minutes,start station name,end station name,usertype,gender_label
0,2020-01-01 00:04:50.192,2020-01-01 00:08:37.037,3.77,Grove St PATH,Newark Ave,Subscriber,Female
1,2020-01-01 00:16:01.670,2020-01-01 00:22:19.080,6.28,Grove St PATH,Brunswick & 6th,Subscriber,Female
2,2020-01-01 00:17:33.877,2020-01-01 00:22:22.442,4.80,Grove St PATH,Brunswick & 6th,Customer,Male
3,2020-01-01 00:32:05.902,2020-01-01 00:39:21.066,7.25,Sip Ave,Astor Place,Customer,Unknown
4,2020-01-01 00:46:19.678,2020-01-01 00:50:11.344,3.85,Grove St PATH,Marin Light Rail,Subscriber,Female


## Step 5: Data Quality Validation Checks
This cell executes multi-dimensional assertions and metrics checks to guarantee there are zero duplicate rows, zero invalid timestamps, and zero mismatched durations.

### Cleaning Phase Summary & Insights
- **Total Raw Records**: 26,020
- **Cleaned Dataset Records**: 25,997 (23 outlier trips removed)
- Outliers were successfully excluded based on unrealistic trip durations.

In [5]:
from pyspark.sql import functions as F

# 1. Check row and column counts
print("Total cleaned rows:", cleaned_df.count())
print("Total columns:", len(cleaned_df.columns))

# 2. Check for duplicate records
print("Duplicate rows:",
      cleaned_df.count() - cleaned_df.distinct().count())

# 3. Check for invalid trip durations
cleaned_df.filter(
    (F.col("duration_minutes") < 1) |
    (F.col("duration_minutes") > 240) |
    F.col("duration_minutes").isNull()
).show()

# 4. Check timestamp inconsistencies
cleaned_df.filter(
    F.col("stoptime") <= F.col("starttime")
).show()

# 5. Check missing values
cleaned_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in cleaned_df.columns
]).show(truncate=False)

Total cleaned rows: 25997
Total columns: 17
Duplicate rows: 0
+------------+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+----------------+------------+
|tripduration|starttime|stoptime|start station id|start station name|start station latitude|start station longitude|end station id|end station name|end station latitude|end station longitude|bikeid|usertype|birth year|gender|duration_minutes|gender_label|
+------------+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+----------------+------------+
+------------+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+-

## Step 6: Save Cleaned Data Locally
We export our cleaned PySpark DataFrame to a local folder in optimized Snappy-compressed Parquet format.

### Validation Check Results
- **Duplicate Rows**: 0
- **Invalid Timestamp Sequences**: 0
- **Timestamp vs. Duration Mismatches**: 0
All validation criteria have passed successfully!

In [6]:
# Save the cleaned DataFrame in Parquet format
output_path = "/content/citibike_processed"

cleaned_df.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Cleaned data saved successfully!")

Cleaned data saved successfully!


## Step 7: Verify Local Saved Parquet
We read back the freshly generated local Parquet files to verify structural completeness and row count consistency.

In [7]:
# Read the saved Parquet data
processed_df = spark.read.parquet(output_path)

print("Original cleaned rows:", cleaned_df.count())
print("Saved rows:", processed_df.count())

processed_df.printSchema()
processed_df.show(5, truncate=False)

Original cleaned rows: 25997
Saved rows: 25997
root
 |-- tripduration: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: integer (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: integer (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- gender_label: string (nullable = true)

+------------+-----------------------+-----------------------+----------------+------------------+----------------------+-----------------------+---------

## Step 8: Install AWS SDK (Boto3)
We install the official AWS SDK library (`boto3`) to facilitate cloud upload interactions.

In [8]:
%pip install -q boto3

## Step 9: Configure AWS Connection
We collect AWS credentials securely using prompt masks and initialize our Amazon S3 client connection to target our project bucket.

In [9]:
import boto3
from getpass import getpass

# Enter your actual keys when Colab prompts you
aws_access_key = getpass("Enter AWS Access Key ID: ")
aws_secret_key = getpass("Enter AWS Secret Access Key: ")

bucket_name = "citi-bike-project-127463484766-us-east-1-an"

s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name="us-east-1"
)

# Test the connection
response = s3.list_objects_v2(
    Bucket=bucket_name,
    MaxKeys=10
)

print("S3 connection successful!")
print("Objects found:", response.get("KeyCount", 0))

Enter AWS Access Key ID: ··········
Enter AWS Secret Access Key: ··········
S3 connection successful!
Objects found: 10


## Step 10: Upload Raw CSV to Amazon S3
This uploads our original raw CSV data into our dedicated S3 bucket under the folder prefix `Raw_data/`.

In [10]:
raw_csv_path = "/content/citibike_data/JC-202001-citibike-tripdata.csv"

s3.upload_file(
    raw_csv_path,
    bucket_name,
    "Raw_data/JC-202001-citibike-tripdata.csv"
)

print("Raw CSV uploaded successfully!")

Raw CSV uploaded successfully!


## Step 11: Upload Processed Parquet to Amazon S3
We iterate through our local Parquet directories and upload each part to S3 under our processed folder structure.

In [11]:
import os

processed_path = "/content/citibike_processed"

for root, dirs, files in os.walk(processed_path):
    for filename in files:
        local_file = os.path.join(root, filename)

        relative_path = os.path.relpath(
            local_file, processed_path
        ).replace(os.sep, "/")

        s3_key = f"processed_data/citibike_jan_2020/{relative_path}"

        s3.upload_file(local_file, bucket_name, s3_key)

print("Processed Parquet files uploaded successfully!")

Processed Parquet files uploaded successfully!


## Step 12: Verify Raw CSV Metadata in Cloud
This queries S3 metadata to verify that both the raw file and processed files exist and shows their exact payload sizes in S3.

In [12]:
# Check the raw CSV
s3.head_object(
    Bucket=bucket_name,
    Key="Raw_data/JC-202001-citibike-tripdata.csv"
)
print("Raw CSV verified in S3!")

# Check processed files
result = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="processed_data/citibike_jan_2020/"
)

print("Processed objects found:", result.get("KeyCount", 0))
for obj in result.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

Raw CSV verified in S3!
Processed objects found: 14
processed_data/citibike_jan_2020/._SUCCESS.crc 8 bytes
processed_data/citibike_jan_2020/.part-00000-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet.crc 6092 bytes
processed_data/citibike_jan_2020/.part-00000-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet.crc 6092 bytes
processed_data/citibike_jan_2020/.part-00000-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet.crc 6092 bytes
processed_data/citibike_jan_2020/.part-00001-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet.crc 552 bytes
processed_data/citibike_jan_2020/.part-00001-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet.crc 552 bytes
processed_data/citibike_jan_2020/.part-00001-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet.crc 552 bytes
processed_data/citibike_jan_2020/_SUCCESS 0 bytes
processed_data/citibike_jan_2020/part-00000-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet 778483 bytes
processed_data/citibike_jan_20

## Step 13: List All Uploaded Files in Amazon S3 Bucket
We retrieve the final object keys, metadata verification, and structure inside our project S3 bucket.

In [13]:
# Verify raw CSV
raw_key = "Raw_data/JC-202001-citibike-tripdata.csv"

s3.head_object(
    Bucket=bucket_name,
    Key=raw_key
)
print("Raw CSV verified in S3!")

# List processed Parquet files
result = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="processed_data/citibike_jan_2020/"
)

print("\nProcessed objects found:", result.get("KeyCount", 0))

for obj in result.get("Contents", []):
    print(obj["Key"], "-", obj["Size"], "bytes")

Raw CSV verified in S3!

Processed objects found: 14
processed_data/citibike_jan_2020/._SUCCESS.crc - 8 bytes
processed_data/citibike_jan_2020/.part-00000-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet.crc - 6092 bytes
processed_data/citibike_jan_2020/.part-00000-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet.crc - 6092 bytes
processed_data/citibike_jan_2020/.part-00000-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet.crc - 6092 bytes
processed_data/citibike_jan_2020/.part-00001-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet.crc - 552 bytes
processed_data/citibike_jan_2020/.part-00001-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet.crc - 552 bytes
processed_data/citibike_jan_2020/.part-00001-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet.crc - 552 bytes
processed_data/citibike_jan_2020/_SUCCESS - 0 bytes
processed_data/citibike_jan_2020/part-00000-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet - 778483 bytes
processed_d

## Step 14: Comprehensive Validation of S3 Data
This cell runs queries to check files, retrieve their properties, and print key characteristics of the uploaded data on S3.

In [14]:

from pyspark.sql import functions as F

# 1. Check row count and column count
print("Rows:", cleaned_df.count())
print("Columns:", len(cleaned_df.columns))

# 2. Check duplicate rows
duplicate_count = (
    cleaned_df.count() - cleaned_df.dropDuplicates().count()
)
print("Duplicate rows:", duplicate_count)

# 3. Check missing values in every column
null_counts = cleaned_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in cleaned_df.columns
])

print("Null values by column:")
null_counts.show(truncate=False)

# 4. Check invalid trip durations
invalid_duration_count = cleaned_df.filter(
    (F.col("duration_minutes") < 1) |
    (F.col("duration_minutes") > 240) |
    F.col("duration_minutes").isNull()
).count()

print("Invalid duration records:", invalid_duration_count)

# 5. Check timestamp validity
invalid_timestamp_count = cleaned_df.filter(
    F.col("starttime").isNull() |
    F.col("stoptime").isNull() |
    (F.col("stoptime") < F.col("starttime"))
).count()

print("Invalid timestamp records:", invalid_timestamp_count)

# 6. Check whether trip duration agrees with timestamps
duration_mismatch_count = cleaned_df.filter(
    F.abs(
        F.col("tripduration") -
        (
            F.col("stoptime").cast("long") -
            F.col("starttime").cast("long")
        )
    ) > 2
).count()

print("Duration/timestamp mismatches (>2 seconds):",
      duration_mismatch_count)


Rows: 25997
Columns: 17
Duplicate rows: 0
Null values by column:
+------------+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+----------------+------------+
|tripduration|starttime|stoptime|start station id|start station name|start station latitude|start station longitude|end station id|end station name|end station latitude|end station longitude|bikeid|usertype|birth year|gender|duration_minutes|gender_label|
+------------+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+----------------+------------+
|0           |0        |0       |0               |0                 |0                     |0                      |0             |0               |0                  

In [15]:
# Verify existing files in S3
raw_key = "Raw_data/JC-202001-citibike-tripdata.csv"

s3.head_object(Bucket=bucket_name, Key=raw_key)
print("Raw CSV exists in S3.")

response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="processed_data/citibike_jan_2020/"
)

parquet_files = [
    item for item in response.get("Contents", [])
    if item["Key"].endswith(".parquet")
]

print("Parquet files found:", len(parquet_files))

for item in parquet_files:
    size_mb = item["Size"] / (1024 * 1024)
    print(f'{item["Key"]} — {size_mb:.2f} MB')

Raw CSV exists in S3.
Parquet files found: 6
processed_data/citibike_jan_2020/part-00000-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet — 0.74 MB
processed_data/citibike_jan_2020/part-00000-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet — 0.74 MB
processed_data/citibike_jan_2020/part-00000-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet — 0.74 MB
processed_data/citibike_jan_2020/part-00001-2402c1f5-f9ce-4532-aee7-eb7306c28499-c000.snappy.parquet — 0.07 MB
processed_data/citibike_jan_2020/part-00001-4933bd47-b63b-4203-b313-2d914a24ee3e-c000.snappy.parquet — 0.07 MB
processed_data/citibike_jan_2020/part-00001-7a89d3cc-ad11-40ce-9b71-456d1a3aa48d-c000.snappy.parquet — 0.07 MB


## Step 15: Run Analytics Summary Queries
We extract key analytics metrics such as trip distribution by hour of day, top stations, user distributions, and demographics of our riders.

In [16]:
from pyspark.sql import functions as F

# 1. Trips by hour of day
trips_by_hour = (
    cleaned_df
    .withColumn("start_hour", F.hour("starttime"))
    .groupBy("start_hour")
    .agg(F.count("*").alias("total_trips"))
    .orderBy("start_hour")
)

print("Trips by hour")
trips_by_hour.show(24, truncate=False)


# 2. Top 10 start stations
top_start_stations = (
    cleaned_df
    .groupBy("start station name")
    .agg(F.count("*").alias("total_trips"))
    .orderBy(F.desc("total_trips"))
    .limit(10)
)

print("Top 10 start stations")
top_start_stations.show(truncate=False)


# 3. Average trip duration by user type
avg_duration_by_user = (
    cleaned_df
    .groupBy("usertype")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("duration_minutes"), 2).alias("avg_duration_minutes")
    )
    .orderBy(F.desc("total_trips"))
)

print("Average duration by user type")
avg_duration_by_user.show(truncate=False)


# 4. Trips by gender label
trips_by_gender = (
    cleaned_df
    .groupBy("gender_label")
    .agg(F.count("*").alias("total_trips"))
    .orderBy(F.desc("total_trips"))
)

print("Trips by gender")
trips_by_gender.show(truncate=False)

Trips by hour
+----------+-----------+
|start_hour|total_trips|
+----------+-----------+
|0         |185        |
|1         |116        |
|2         |42         |
|3         |33         |
|4         |71         |
|5         |298        |
|6         |857        |
|7         |1916       |
|8         |3084       |
|9         |1583       |
|10        |1034       |
|11        |1039       |
|12        |1147       |
|13        |1156       |
|14        |1041       |
|15        |1165       |
|16        |1592       |
|17        |2674       |
|18        |2743       |
|19        |1663       |
|20        |1054       |
|21        |755        |
|22        |475        |
|23        |274        |
+----------+-----------+

Top 10 start stations
+--------------------------+-----------+
|start station name        |total_trips|
+--------------------------+-----------+
|Grove St PATH             |3099       |
|Sip Ave                   |1491       |
|Hamilton Park             |1325       |
|Columbus Dr at E

## Step 16: Partition Parquet for Efficient Lakehouse Queries
We partition our PySpark DataFrame on disk by transaction Year and Month and save the partitioned parquet files locally.

### Summary Insights
- **Peak Demand Hours**: Major rush hours occur at 8:00 AM (3,084 trips) and 5:00 PM - 6:00 PM (over 5,400 trips collectively).
- **Top Starting Hub**: Grove St PATH station is by far the most popular departure point with over 3,099 trips.
- **Demographics & Usertype**: Subscribers represent the massive majority (24,427 trips) compared to single-day customers (1,570 trips). Males made up over 71% of riders.

In [17]:
from pyspark.sql import functions as F

# Add partition columns from the trip start timestamp
partitioned_df = (
    cleaned_df
    .withColumn("year", F.year("starttime"))
    .withColumn("month", F.month("starttime"))
)

# Save partitioned Parquet locally in Colab
partitioned_path = "/content/citibike_partitioned"

(
    partitioned_df.write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(partitioned_path)
)

print("Partitioned Parquet created successfully.")
print("Rows:", partitioned_df.count())
print("Partition columns: year, month")

Partitioned Parquet created successfully.
Rows: 25997
Partition columns: year, month


## Step 17: Verify Partition Schema
This reads back our partitioned parquet files to verify that Spark reads the virtual partition directories as structural partition columns.

In [18]:
# Read back and verify partitioned Parquet

partitioned_check = spark.read.parquet(partitioned_path)

print("Rows read back:", partitioned_check.count())
partitioned_check.select("year", "month").distinct().show()
partitioned_check.printSchema()

Rows read back: 25997
+----+-----+
|year|month|
+----+-----+
|2020|    1|
+----+-----+

root
 |-- tripduration: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: integer (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: integer (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- gender_label: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



### Partition Validation Result
- **Successful Readback**: Verified that the partitioned Parquet files load correctly.
- **Output Partitions**: Successfully resolved virtual directory structures into physical DataFrame columns (`year = 2020` and `month = 1`).

## Step 18: Build Final Analytics Summary & Write to CSV
We aggregate the global key-performance metrics of our dataset into a compact, single-row summary report and write it as a final CSV dataset.

In [19]:
from pyspark.sql import functions as F

# Create a compact analytics summary
analytics_summary = cleaned_df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("duration_minutes"), 2).alias("avg_duration_minutes"),
    F.round(F.min("duration_minutes"), 2).alias("min_duration_minutes"),
    F.round(F.max("duration_minutes"), 2).alias("max_duration_minutes"),
    F.countDistinct("start station name").alias("unique_start_stations"),
    F.countDistinct("end station name").alias("unique_end_stations")
)

analytics_summary.show(truncate=False)

# Top 10 start stations
top_start_stations.show(truncate=False)

# Save the analytics summary locally as CSV
analytics_summary.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv("/content/citibike_analytics_summary")

print("Analytics summary saved to /content/citibike_analytics_summary")

+-----------+--------------------+--------------------+--------------------+---------------------+-------------------+
|total_trips|avg_duration_minutes|min_duration_minutes|max_duration_minutes|unique_start_stations|unique_end_stations|
+-----------+--------------------+--------------------+--------------------+---------------------+-------------------+
|25997      |7.69                |1.02                |233.13              |51                   |53                 |
+-----------+--------------------+--------------------+--------------------+---------------------+-------------------+

+--------------------------+-----------+
|start station name        |total_trips|
+--------------------------+-----------+
|Grove St PATH             |3099       |
|Sip Ave                   |1491       |
|Hamilton Park             |1325       |
|Columbus Dr at Exchange Pl|1151       |
|Harborside                |1089       |
|Newport PATH              |987        |
|Marin Light Rail          |902    

#### 1. Compact Analytics Summary
- **Total Trips Analyzed**: 25,997
- **Average Trip Duration**: 7.69 minutes
- **Minimum Trip Duration**: 1.02 minutes
- **Maximum Trip Duration**: 233.13 minutes
- **Unique Departure Stations**: 51
- **Unique Arrival Stations**: 53

#### 2. Top 10 Start Stations
| Rank | Start Station Name | Total Trips |
|---|---|---|
| 1 | Grove St PATH | 3,099 |
| 2 | Sip Ave | 1,491 |
| 3 | Hamilton Park | 1,325 |
| 4 | Columbus Dr at Exchange Pl | 1,151 |
| 5 | Harborside | 1,089 |
| 6 | Newport PATH | 987 |
| 7 | Marin Light Rail | 902 |
| 8 | Brunswick & 6th | 776 |
| 9 | City Hall | 745 |
| 10 | Newark Ave | 720 |

# Citi Bike Data Engineering Pipeline

## Project Overview
Built a PySpark data engineering pipeline to process Citi Bike trip data for January 2020. The project demonstrates ingestion, data cleaning, validation, Parquet storage, partitioning, and business analytics using Google Colab and Amazon S3.

## Technology Stack
- Python and PySpark
- Apache Spark
- Amazon S3
- Parquet
- Google Colab

## Data Pipeline
1. Downloaded the January 2020 Citi Bike CSV dataset.
2. Loaded the raw data into a Spark DataFrame.
3. Converted trip timestamps and calculated trip duration in minutes.
4. Filtered trips to a duration range of 1–240 minutes.
5. Removed records missing essential station names.
6. Validated row counts, duplicate records, null values, and timestamp/duration consistency.
7. Stored the raw CSV and processed Parquet files in Amazon S3.
8. Created partitioned Parquet output using year and month.
9. Analyzed hourly trip patterns, popular stations, user types, and gender labels.

## Validation Results
- Raw records loaded: 26,020
- Cleaned records: 25,997
- Records removed: 23
- Cleaned columns: 17
- Duplicate rows detected: 0
- Invalid duration records: 0
- Invalid timestamp records: 0
- Partition verified: year=2020, month=1

## Business Insights
- Subscriber trips were more frequent than customer trips in the cleaned dataset.
- Customer trips had a higher average duration than subscriber trips.
- Trip activity varied by hour, with clear differences between overnight and daytime activity.

## Important Note
This project analyzes one month of Jersey City trip data. The cleaning rules and results apply to this dataset and should not be generalized to the entire Citi Bike system.

In [20]:
# Final Pipeline Status

print("Citi Bike Data Engineering Pipeline")
print("-" * 40)

print("Raw dataset rows:", df.count())
print("Cleaned dataset rows:", cleaned_df.count())
print("Cleaned dataset columns:", len(cleaned_df.columns))

print("Canonical S3 processed path:")
print(f"s3://{bucket_name}/processed_data/citibike_jan_2020/")

print("Pipeline status: COMPLETED")

Citi Bike Data Engineering Pipeline
----------------------------------------
Raw dataset rows: 26020
Cleaned dataset rows: 25997
Cleaned dataset columns: 17
Canonical S3 processed path:
s3://citi-bike-project-127463484766-us-east-1-an/processed_data/citibike_jan_2020/
Pipeline status: COMPLETED
